In [3]:
import logging

from langchain_core.documents import Document

from langchain_huggingface import (
    HuggingFaceEmbeddings,
    HuggingFaceEndpoint,
    ChatHuggingFace
)

from langchain_chroma import Chroma

from langchain_classic.retrievers import MultiQueryRetriever

c:\Users\disha\OneDrive\Desktop\Langchain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# 1. Sample documents
documents = [
    Document(page_content="Python is a high-level, interpreted programming language known for its readability and simple syntax.", metadata={"source": "doc1"}),
    Document(page_content="Machine learning is a subset of AI that enables systems to learn patterns from data without explicit programming.", metadata={"source": "doc2"}),
    Document(page_content="Deep learning uses multi-layered neural networks to model complex patterns in data.", metadata={"source": "doc3"}),
    Document(page_content="The Great Barrier Reef is the world's largest coral reef system, located off Queensland, Australia.", metadata={"source": "doc4"}),
    Document(page_content="Climate change refers to long-term shifts in temperatures and weather patterns due to human activity.", metadata={"source": "doc5"}),
    Document(page_content="Natural language processing enables computers to understand, interpret, and generate human language.", metadata={"source": "doc6"}),
]


In [5]:
# 2. Embeddings + vector store
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)
vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2597.34it/s]


In [6]:
# 3. LLM used to generate query variations
from dotenv import load_dotenv

load_dotenv()
llm_endpoint = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    huggingfacehub_api_token="huggingfacehub_api_token",
    max_new_tokens=256,
    temperature=0.3
)
llm = ChatHuggingFace(llm=llm_endpoint)

In [27]:
# 4. Build the MultiQueryRetriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
)

In [28]:
# 5. Query

query = "How does AI relate to understanding language?"

results = multi_query_retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(doc.page_content)
    print(doc.metadata)
    print()

--- Result 1 ---
Natural language processing enables computers to understand, interpret, and generate human language.
{'source': 'doc6'}

--- Result 2 ---
Natural language processing enables computers to understand, interpret, and generate human language.
{'source': 'doc6'}

--- Result 3 ---
Deep learning uses multi-layered neural networks to model complex patterns in data.
{'source': 'doc3'}

--- Result 4 ---
Machine learning is a subset of AI that enables systems to learn patterns from data without explicit programming.
{'source': 'doc2'}

